# Assignment 2 — California Housing: EDA + Multiple Linear Regression
**Data Science Lab (CS2311)**

This one notebook covers all three questions, because each builds on the previous one:
1. **Q1** — EDA (dimensions, types, statistics, missing values, duplicates, outliers, correlations) and preprocessing (outliers, scaling, train/test split).
2. **Q2** — Three Multiple Linear Regression models (A: top-3 correlated features, B: 5 chosen features, C: all features), compared on MAE, MSE, RMSE, R² and training time.
3. **Q3** — Feature selection (correlation filter, coefficient magnitude, RFE), retrain, and compare with the all-feature model.

**Intuition for linear regression:** we price a house with a weighted formula

    price ≈ b0 + b1·MedInc + b2·HouseAge + ... + b8·Longitude

The model learns the weights b0…b8 that make the predictions as close as possible to the real prices (by minimising the sum of squared errors).

In [ ]:
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.feature_selection import RFE
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', None)
RANDOM_STATE = 42

## Loading the data

`fetch_california_housing()` downloads the dataset the first time (needs internet) and caches it.

**Fallback:** if the download is blocked, we rebuild the *same* 8 features from the raw 1990 census CSV on GitHub (the Kaggle "housing.csv"). The raw file stores totals per block, so we divide by the number of households to get averages, exactly as scikit-learn does. The raw file also has 207 missing `total_bedrooms` values, which gives us real missing values to handle.

In [ ]:
def load_california():
    try:
        data = fetch_california_housing(as_frame=True)
        print('Loaded from scikit-learn.')
        return data.frame
    except Exception as e:
        print('scikit-learn download failed:', type(e).__name__, '→ using the GitHub CSV fallback.')
        url = 'https://raw.githubusercontent.com/ageron/handson-ml2/master/datasets/housing/housing.csv'
        raw = pd.read_csv(url)
        return pd.DataFrame({
            'MedInc':      raw['median_income'],
            'HouseAge':    raw['housing_median_age'],
            'AveRooms':    raw['total_rooms'] / raw['households'],
            'AveBedrms':   raw['total_bedrooms'] / raw['households'],
            'Population':  raw['population'],
            'AveOccup':    raw['population'] / raw['households'],
            'Latitude':    raw['latitude'],
            'Longitude':   raw['longitude'],
            'MedHouseVal': raw['median_house_value'] / 100_000,   # same unit as sklearn: $100,000s
        })

df = load_california()
df.head()

**Attribute meanings** (each row is one census *block group*, a small neighbourhood):

| Column | Meaning |
|---|---|
| **MedInc** | Median household income (in tens of thousands of dollars) |
| **HouseAge** | Median age of the houses (years) |
| **AveRooms** | Average rooms per household |
| **AveBedrms** | Average bedrooms per household |
| **Population** | Number of people in the block group |
| **AveOccup** | Average people per household |
| **Latitude / Longitude** | Location |
| **MedHouseVal (target)** | Median house value, in units of $100,000 (capped at 5.0) |

---
# Question 1 — Exploratory Data Analysis
## 1.1 Dimensions, feature types and summary statistics

In [ ]:
print('Shape (rows, columns):', df.shape)
print()
df.info()

In [ ]:
df.describe().T.round(3)

**What to notice in `describe()`:**
- `AveRooms`, `AveBedrms`, `AveOccup` and `Population` have a **max far above the 75th percentile** → extreme outliers (some block groups are hotels or dormitories).
- `MedHouseVal` max is exactly 5.00001 → values were **capped** at $500,000.
- All columns are numeric (`float64`), so no encoding is needed.

## 1.2 Missing values and duplicates

In [ ]:
missing = df.isnull().sum()
print('Missing values per column:\n', missing, sep='')
print('\nTotal duplicate rows:', df.duplicated().sum())

## 1.3 Distributions — histograms

In [ ]:
df.hist(bins=50, figsize=(16, 10), edgecolor='black')
plt.suptitle('Distribution of every attribute', fontsize=16)
plt.tight_layout()
plt.show()

**Insights:** `MedInc` and the target are right-skewed; the target has a spike at 5.0 (the cap); `HouseAge` has a spike at 52 (also capped); the "Ave" columns and `Population` are squeezed into one bar because a few huge outliers stretch the x-axis.

## 1.4 Outliers — box plots and the IQR rule

**IQR rule:** IQR = Q3 − Q1. A value is an outlier if it lies below Q1 − 1.5·IQR or above Q3 + 1.5·IQR.

Dry run: Q1 = 4, Q3 = 6 → IQR = 2 → fences are 1 and 9. A value of 140 is an outlier; 5 is not.

In [ ]:
features = df.columns.drop('MedHouseVal')
fig, axes = plt.subplots(2, 4, figsize=(18, 7))
for ax, col in zip(axes.ravel(), features):
    sns.boxplot(x=df[col], ax=ax, color='skyblue')
    ax.set_title(col)
plt.suptitle('Box plots (dots beyond the whiskers are outliers)', fontsize=15)
plt.tight_layout()
plt.show()

In [ ]:
def iqr_bounds(series):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

outlier_counts = {}
for col in features:
    low, high = iqr_bounds(df[col])
    outlier_counts[col] = int(((df[col] < low) | (df[col] > high)).sum())
pd.Series(outlier_counts, name='IQR outliers').to_frame()

## 1.5 Correlations — heatmap and scatter plots

**Correlation (Pearson r):** a number from −1 to +1 measuring how strongly two columns move together in a straight line. +1 = perfectly together, 0 = no linear relation, −1 = perfectly opposite.

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0, square=True)
plt.title('Correlation heatmap')
plt.show()

target_corr = df.corr()['MedHouseVal'].drop('MedHouseVal').sort_values(key=abs, ascending=False)
print('Correlation of each feature with MedHouseVal (sorted by strength):')
print(target_corr.round(3))

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 5))
sns.scatterplot(x='MedInc', y='MedHouseVal', data=df, alpha=0.15, s=10, ax=ax[0])
ax[0].set_title('Income vs house value (strongest relation)')
sns.scatterplot(x='AveRooms', y='MedHouseVal', data=df[df.AveRooms < 15], alpha=0.15, s=10, ax=ax[1])
ax[1].set_title('Average rooms vs house value (AveRooms < 15)')
sc = ax[2].scatter(df.Longitude, df.Latitude, c=df.MedHouseVal, cmap='viridis', s=3)
plt.colorbar(sc, ax=ax[2], label='MedHouseVal')
ax[2].set_title('Location map: expensive houses cluster on the coast')
ax[2].set_xlabel('Longitude'); ax[2].set_ylabel('Latitude')
plt.tight_layout()
plt.show()

**EDA insights:**
1. `MedInc` is by far the strongest predictor of house value (r ≈ 0.69).
2. `AveRooms` and `AveBedrms` are strongly correlated **with each other** (multicollinearity) — they carry overlapping information.
3. `Latitude` and `Longitude` are strongly negatively correlated with each other (California runs diagonally), and each has a weak correlation with the target **on its own**; together they mark the expensive coastal areas (Bay Area, Los Angeles).
4. The horizontal line at 5.0 in the income scatter is the price cap.

---
## 1.6 Preprocessing

**Step 1 — Missing values:** fill with the **median** (robust to the outliers we found). If the data came from scikit-learn there are none and this step changes nothing.

**Step 2 — Outliers:** we **cap (clip)** extreme values at the IQR fences instead of deleting rows. Deleting would throw away the other, perfectly good columns of those rows. We cap only the four columns with unrealistic extremes; `MedInc` outliers are genuine rich neighbourhoods, so we keep them.

**Step 3 — Remove exact duplicates** (if any).

In [ ]:
df_clean = df.copy()

for col in df_clean.columns:
    if df_clean[col].isnull().any():
        median = df_clean[col].median()
        df_clean[col] = df_clean[col].fillna(median)
        print(f'Filled {col} missing values with median = {median:.4f}')

cap_cols = ['AveRooms', 'AveBedrms', 'Population', 'AveOccup']
for col in cap_cols:
    low, high = iqr_bounds(df_clean[col])
    df_clean[col] = df_clean[col].clip(lower=low, upper=high)
    print(f'{col:11s} capped to [{low:.2f}, {high:.2f}]')

df_clean = df_clean.drop_duplicates()
print('\nShape after cleaning:', df_clean.shape)
print('Missing values left:', df_clean.isnull().sum().sum())

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 3.5))
for ax, col in zip(axes, cap_cols):
    sns.boxplot(x=df_clean[col], ax=ax, color='lightgreen')
    ax.set_title(f'{col} after capping')
plt.tight_layout()
plt.show()

**Step 4 — Train/test split (80/20):** the model learns from the training set; the test set is kept hidden and used only for the final evaluation, simulating new, unseen houses.

**Step 5 — Feature scaling (StandardScaler):** z = (x − mean) / std, so every feature has mean 0 and standard deviation 1.
- Plain linear regression does not *need* scaling to make correct predictions, but scaling makes the **coefficients comparable** (we use them for feature selection in Q3), and it is required for regularized models and RFE rankings.
- **Fit the scaler on the training set only**, then apply (`transform`) it to the test set. Fitting on the test set would leak information about the "unseen" data into training (**data leakage**).

In [ ]:
X = df_clean.drop('MedHouseVal', axis=1)     # axis=1 → drop a COLUMN
y = df_clean['MedHouseVal']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE)

scaler = StandardScaler()
X_train_s = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns, index=X_train.index)
X_test_s  = pd.DataFrame(scaler.transform(X_test),      columns=X.columns, index=X_test.index)

print('Train:', X_train_s.shape, ' Test:', X_test_s.shape)
X_train_s.describe().loc[['mean', 'std']].round(3)

After scaling, every training column has mean ≈ 0 and std ≈ 1. ✔

---
# Question 2 — Three Multiple Linear Regression models

## Evaluation metrics

| Metric | Formula (plain text) | Meaning | Better |
|---|---|---|---|
| **MAE** | mean of abs(y − ŷ) | Average size of the error, in target units ($100k) | Lower |
| **MSE** | mean of (y − ŷ)² | Squares errors, so big mistakes are punished heavily | Lower |
| **RMSE** | sqrt(MSE) | Like MAE but punishes big errors; back in target units | Lower |
| **R²** | 1 − SS_res / SS_tot | Fraction of the variance in prices that the model explains | Higher (max 1) |

Dry run: y = [3, 5], ŷ = [2, 7] → errors [1, −2] → MAE = 1.5, MSE = 2.5, RMSE = 1.58.

In [ ]:
def evaluate_model(name, features, X_tr, X_te, y_tr, y_te, repeats=20):
    """Fit LinearRegression on the given features; return metrics + average training time."""
    times = []
    for _ in range(repeats):                # repeat: one fit takes milliseconds, so averaging gives a stable time
        model = LinearRegression()
        start = time.perf_counter()
        model.fit(X_tr[features], y_tr)
        times.append(time.perf_counter() - start)
    pred = model.predict(X_te[features])
    mse = mean_squared_error(y_te, pred)
    return {
        'Model': name,
        '#Features': len(features),
        'Features': ', '.join(features),
        'MAE': mean_absolute_error(y_te, pred),
        'MSE': mse,
        'RMSE': np.sqrt(mse),
        'R2': r2_score(y_te, pred),
        'Train time (ms)': np.mean(times) * 1000,
    }, model, pred

## Choosing the three feature sets

- **Model A:** the 3 features with the highest |correlation| with the target (computed on the training data).
- **Model B:** 5 features chosen from our analysis: `MedInc` (strongest predictor), `HouseAge` and `AveRooms` (property characteristics), plus `Latitude` and `Longitude` **together** (location — individually weak, jointly very informative, as the map showed).
- **Model C:** all 8 features.

In [ ]:
train_corr = (pd.concat([X_train, y_train], axis=1).corr()['MedHouseVal']
              .drop('MedHouseVal').abs().sort_values(ascending=False))
print('|Correlation| with target on TRAINING data:\n', train_corr.round(3), sep='')

feature_sets = {
    'Model A (top-3 correlated)': train_corr.index[:3].tolist(),
    'Model B (5 selected)':       ['MedInc', 'HouseAge', 'AveRooms', 'Latitude', 'Longitude'],
    'Model C (all features)':     X.columns.tolist(),
}
for k, v in feature_sets.items():
    print(f'{k}: {v}')

In [ ]:
rows, models, preds = [], {}, {}
for name, feats in feature_sets.items():
    res, mdl, pred = evaluate_model(name, feats, X_train_s, X_test_s, y_train, y_test)
    rows.append(res); models[name] = mdl; preds[name] = pred

results_q2 = pd.DataFrame(rows).set_index('Model')
results_q2.drop(columns='Features').round(4)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 4.5))
results_q2[['MAE', 'RMSE']].plot(kind='bar', ax=ax[0], rot=15)
ax[0].set_title('Error metrics (lower is better)')
results_q2['R2'].plot(kind='bar', ax=ax[1], rot=15, color='tab:green')
ax[1].set_title('R² (higher is better)')
ax[1].set_ylim(0, 1)
results_q2['Train time (ms)'].plot(kind='bar', ax=ax[2], rot=15, color='tab:purple')
ax[2].set_title('Training time (ms)')
plt.tight_layout()
plt.show()

In [ ]:
best_name = results_q2['R2'].idxmax()
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
ax[0].scatter(y_test, preds[best_name], alpha=0.2, s=8)
ax[0].plot([0, 5], [0, 5], 'r--', label='Perfect prediction')
ax[0].set_xlabel('Actual MedHouseVal'); ax[0].set_ylabel('Predicted')
ax[0].set_title(f'Actual vs predicted — {best_name}')
ax[0].legend()
residuals = y_test - preds[best_name]
sns.histplot(residuals, bins=50, kde=True, ax=ax[1])
ax[1].set_title('Residuals (actual − predicted)')
plt.tight_layout()
plt.show()

In [ ]:
coef_c = pd.Series(models['Model C (all features)'].coef_, index=X.columns).sort_values()
coef_c.plot(kind='barh', figsize=(8, 4), color=['tab:red' if c < 0 else 'tab:blue' for c in coef_c])
plt.title('Model C coefficients (on scaled features)')
plt.xlabel('Change in MedHouseVal for +1 std of the feature')
plt.show()

## Q2 Discussion

- **Best feature combination:** Model C (all features) gives the highest R² and the lowest errors. Model B is close behind; Model A is clearly worst.
- **Why A is weak:** Model A has no location information. Its extra features add little *beyond* income. Model B adds `Latitude` **and** `Longitude` together, which encode location, the second biggest driver of price. Correlation with the target alone is not a perfect guide to usefulness: Longitude has almost zero correlation with price on its own, yet it is essential when paired with Latitude.
- **Coefficients:** income has the largest positive effect; `Latitude` and `Longitude` have large negative coefficients (moving north or east, away from the coast, lowers price).
- **Time vs features vs accuracy:** training time grows with the number of features, but for linear regression on ~16k rows it stays in **milliseconds** (the exact numbers vary run to run). Here the extra features cost almost nothing and buy accuracy, so using more informative features is worth it. On very large datasets (millions of rows, thousands of features) the cost grows and fewer, well-chosen features also reduce overfitting risk.

---
# Question 3 — Feature selection

Three ways to pick the most relevant predictors:

1. **Correlation filter:** keep features with |r| with the target above a threshold. Fast, but looks at each feature alone and ignores interactions (it would miss the Latitude + Longitude pair).
2. **Coefficient magnitude:** on *scaled* data, a bigger |coefficient| means more influence on the prediction.
3. **RFE (Recursive Feature Elimination):** fit the model → drop the feature with the smallest |coefficient| → refit → repeat until k features remain.

**RFE dry run (k = 2, features a, b, c, d):** fit on [a, b, c, d], smallest |coef| is c → drop c. Fit on [a, b, d], smallest is a → drop a. Result: [b, d].

In [ ]:
# Method 1: correlation filter (threshold 0.1)
corr_selected = train_corr[train_corr > 0.1].index.tolist()
print('Correlation filter (|r| > 0.1):', corr_selected)

# Method 2: coefficient magnitude from the full model
coef_rank = coef_c.abs().sort_values(ascending=False)
print('\nFeatures ranked by |coefficient| (scaled data):')
print(coef_rank.round(4))

# Method 3: RFE for every possible k, to see how R² changes with the number of features
rfe_rows = []
for k in range(1, len(X.columns) + 1):
    rfe = RFE(LinearRegression(), n_features_to_select=k).fit(X_train_s, y_train)
    feats = X.columns[rfe.support_].tolist()
    res, _, _ = evaluate_model(f'RFE k={k}', feats, X_train_s, X_test_s, y_train, y_test, repeats=5)
    rfe_rows.append(res)
rfe_table = pd.DataFrame(rfe_rows).set_index('Model')
rfe_table[['#Features', 'Features', 'RMSE', 'R2']].round(4)

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(rfe_table['#Features'], rfe_table['R2'], marker='o')
plt.xlabel('Number of features kept by RFE')
plt.ylabel('Test R²')
plt.title('Accuracy vs number of selected features (RFE)')
plt.grid(True)
plt.show()

rfe5 = RFE(LinearRegression(), n_features_to_select=5).fit(X_train_s, y_train)
print('RFE ranking (1 = selected):')
print(pd.Series(rfe5.ranking_, index=X.columns).sort_values())

## Retrain on the selected features and compare with all features

We keep the **5 features chosen by RFE** (the elbow of the curve above, where adding more features gives only small gains) and compare with the full model and the correlation-filter model.

In [ ]:
selected = X.columns[rfe5.support_].tolist()
comparison_rows = []
for name, feats in {
    'All features (8)': X.columns.tolist(),
    f'RFE-selected ({len(selected)})': selected,
    f'Correlation filter ({len(corr_selected)})': corr_selected,
}.items():
    res, _, _ = evaluate_model(name, feats, X_train_s, X_test_s, y_train, y_test)
    comparison_rows.append(res)

comparison_q3 = pd.DataFrame(comparison_rows).set_index('Model')
comparison_q3[['#Features', 'Features', 'MAE', 'MSE', 'RMSE', 'R2', 'Train time (ms)']].round(4)

In [ ]:
comparison_q3[['MAE', 'RMSE', 'R2']].plot(kind='bar', figsize=(10, 4.5), rot=0)
plt.title('All features vs selected features')
plt.show()

delta = comparison_q3['R2'].iloc[1] - comparison_q3['R2'].iloc[0]
print('RFE kept   :', selected)
print('RFE dropped:', [c for c in X.columns if c not in selected])
print(f'Change in R² when going from 8 to {len(selected)} features (RFE): {delta:+.4f}')
verdict = ('maintained (change < 0.02)' if abs(delta) < 0.02
           else 'improved' if delta > 0 else 'degraded')
print('Prediction accuracy was', verdict)

## Q3 Conclusion

- RFE keeps income and **both** location features (see the printed list above), and drops features with small coefficients such as `Population`. Notice that RFE with k = 1 picks `Latitude`, not `MedInc`: RFE ranks by coefficient size inside the full model, not by individual correlation.
- Reducing from 8 to 5 features changes R² only slightly (see the printed change above): accuracy is **maintained** with a simpler model. The correlation filter, which ignores how features work together and leaves out Longitude, performs worse.
- **Most suitable feature set:** if we want the very best accuracy, all 8 features. If we value simplicity and interpretability with almost the same accuracy, the RFE-selected 5. Reducing features did not *improve* accuracy here because linear regression on 16k rows is not overfitting; its benefit is a simpler, faster, easier-to-explain model.
- **Limitation:** R² ≈ 0.6 means linear regression explains about 60% of the variation. Prices depend on location in a non-linear way, which tree-based models capture better.